# Metrics Harness: Implementing Precision@K, Recall@K, and NDCG@K (MovieLens 1M)

After preparing the **MovieLens 1M* dataset with clean temporal splits and leak-free train/val/test partitions, our next step is to **evaluate recommendation quality**.<br>
This notebook introduces a reusable **metrics harness** for ranking evaluation — computing Precision@K, Recall@K, and NDCG@K over the validation and test splits you built earlier.

## Connecting to the Real World
n production recommendation systems — like **Netflix**, **Spotify**, or **Amazon** — success isn’t about predicting ratings; it’s about ranking.
The key business question is simple:

> “Are we showing the right items at the top for each user?”

When you open Netflix, you don’t scroll through all 100 options — you pick from the first few.
That’s why top-K metrics matter: they tell us how well our system surfaces relevant content early in the list.
- **Precision@K** checks how many of the top-K recommendations were correct.
- **Recall@K** checks how many of the true relevant items appeared in those top-K.
- **NDCG@K** measures how well the ranking orders relevant items — rewarding systems that put them higher up.

By using these metrics, we can evaluate **popularity baselines, collaborative filtering, matrix factorization, and learning-to-rank models** in a consistent and meaningful way.


## What We’re Achieving Here
- Implement a **standard evaluation** framework for all downstream recommender models.
- Compute user-level Precision@K, Recall@K, and NDCG@K, then macro-average across the population.
- Ensure metrics align with the **temporal, implicit-feedback setup** established in the dataset prep notebook.
- Enable fair, comparable evaluation for future experiments and leaderboard tracking.

## Key Takeaways
- Understand the **role of top-K metrics** in offline recommender evaluation.
- Learn to compute metrics per user and aggregate across validation/test splits.
- Build a **modular, reusable metrics harness** for future notebooks.
- Establish consistency in metric definitions across baselines, MF, and LTR pipelines.

## Overall Agenda
1. **Configuration** – import splits, define K values, and seed setup
1. **Metric Intuition** – conceptual walkthrough of Precision, Recall, and NDCG
1. **Per-User Metric Functions** – implement precision_at_k, recall_at_k, ndcg_at_k
1. **Batch Evaluation Utility** – macro-average metrics across all users and K values
1. **Sanity Tests** – verify logic using small examples with known outcomes
1. **Validation Run** – apply metrics to your MovieLens val/test splits
1. **Metric Summary & Next Steps** – summarize performance and prepare for baseline modeling

## 1. Configuration

### 1.1 Objective

Set up all required parameters, imports, and environment settings for evaluating our MovieLens 1M recommendations.
This step ensures consistency across future notebooks and reproducibility of results.

### 1.2 Why This Step Matters

Before computing ranking metrics, we need to:
1. Load the **validation and test splits** generated earlier.
1. Define the **K values** at which metrics will be evaluated.
1. Initialize random seeds for reproducibility.
1. Create placeholder structures for predictions and ground truth data.

This setup acts as the foundation for every evaluation — whether you’re testing a baseline model or a deep learning recommender later.

### 1.3 Implementation Steps

#### 1.3.1 Import Dependencies

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: mount failed

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Display config for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [ ]:
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/upgrad_live_sessions/"
    "Recommendation_systems/notebook-1/C6"
)

In [ ]:
DATA_SPLITS_DIR = PROJECT_ROOT/"data_splits"

In [ ]:
assert PROJECT_ROOT.exists() and PROJECT_ROOT.is_dir()
print("✅ PROJECT_ROOT is valid:", PROJECT_ROOT)

In [ ]:
assert DATA_SPLITS_DIR.exists() and DATA_SPLITS_DIR.is_dir()
print("✅ DATA_SPLITS_DIR is valid:", DATA_SPLITS_DIR)

#### 1.3.2 Define Evaluation Parameters

In [ ]:
# Values of K at which we’ll compute Precision, Recall, and NDCG
K_LIST = [5, 10, 20]

# Random seed for reproducibility (used later for any sampling)
SEED = 42
np.random.seed(SEED)

print(f"Configured K values: {K_LIST}")
print(f"Random seed set to: {SEED}")

#### 1.3.3 Load Validation and Test Splits

In [ ]:
# Assuming the previous notebook exported these splits
TIMESTAMP = "20251103_1428"
# VAL_PATH  = f"DATA_SPLITS_DIR/val_{TIMESTAMP}.csv"
# TEST_PATH = f"DATA_SPLITS_DIR/test_{TIMESTAMP}.csv"

VAL_PATH  = DATA_SPLITS_DIR / f"val_{TIMESTAMP}.csv"
TEST_PATH = DATA_SPLITS_DIR / f"test_{TIMESTAMP}.csv"

# Load splits into DataFrames
val_df  = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

#### 1.3.4 Create User → Ground Truth Map

In [ ]:
# Build dictionary of true positive interactions for each user
user_to_truth_val = (
    val_df.groupby("UserID")["MovieID"].apply(set).to_dict()
)
user_to_truth_test = (
    test_df.groupby("UserID")["MovieID"].apply(set).to_dict()
)

print(f"Users in validation: {len(user_to_truth_val):,}")
print(f"Users in test: {len(user_to_truth_test):,}")

#### 1.3.5 Placeholder for Model Predictions

In [ ]:
# Placeholder for predictions; to be replaced with model outputs later
# Format: {user_id: [ranked_item1, ranked_item2, ...]}

user_to_preds_val = defaultdict(list)
user_to_preds_test = defaultdict(list)

print("Prediction dictionaries initialized.")

## 2. Metric Intuition & Expected Behavior

### 2.1 Objective

Before implementing any formulas, we’ll build intuition for how **Precision@K, Recall@K, and NDCG@K** behave — both conceptually and through simple examples.
This ensures learners don’t just code metrics mechanically, but understand what each one measures and when to use it.

### 2.2 Why This Step Matters

Offline recommender evaluation isn’t about accuracy or RMSE anymore — it’s about **ranking quality**.
In production, the order in which we show items drives engagement, conversions, or retention.

Ranking metrics tell us:
1. Are the **top few** items relevant?
1. Do we recover **most of the true interests** a user has?
1. Are the **most relevant items placed higher** in the recommendation list?

Understanding these questions conceptually prepares us to interpret results later — not just compute them.

### 2.3 Conceptual Overview

#### 2.3.1 Precision@K

**Definition:** Fraction of recommended items in the top-K that are relevant.

$$
\text{Precision@K} = \frac{\text{\# of relevant items in top-K}}{K}
$$

**Example:**
If top-5 recommended movies are `[12, 45, 89, 32, 50]`

and the user actually liked `{45, 50, 70}`,

then relevant = 2 (45, 50) → **Precision@5 = 2/5** = 0.4


Precision focuses on quality — how accurate the top-K recommendations are.

#### 2.3.2 Recall@K

**Definition:** Fraction of relevant items that appear in the top-K list.

$$\text{Recall@K} = \frac{\text{\# of relevant items in top-K}}{\text{Total \# of relevant items}}$$

**Example:**
Same user with `{45, 50, 70}` relevant and 2 retrieved (45, 50) → **Recall@5 = 2/3** ≈ 0.67

Recall focuses on **coverage** — how much of the user’s actual interests were captured.

#### 2.3.3 NDCG@K (Normalized Discounted Cumulative Gain)

**Definition:** Measures how well-ranked the relevant items are, discounting lower positions.

$$DCG@K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}$$

and

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

where _rel_i = 1_ if the item at rank i is relevant.

**Example:**
If the correct movies `{45, 50}` appear at positions `[2, 5]` in the ranked list,
then

$$DCG@5 = \frac{1}{\log_2(3)} + \frac{1}{\log_2(6)} ≈ 0.63 + 0.39 = 1.02$$

and if both were ideally at top `[1, 2]`,

$$IDCG@5 = 1 + \frac{1}{\log_2(3)} ≈ 1.63$$
**→ NDCG@5 = 1.02 / 1.63 ≈ 0.63**

NDCG rewards ranking the relevant items higher —
capturing both correctness and ordering.

### 2.4 Intuitive Comparison Table

| Metric | What It Measures | High When | Ideal Use Case |
|:--|:--|:--|:--|
| **Precision@K** | Accuracy of top-K items | Most top-K are relevant | Ensuring recommendations are not spammy |
| **Recall@K** | Coverage of relevant items | Most relevant items retrieved | Maximizing discovery and breadth |
| **NDCG@K** | Ranking quality | Relevant items ranked near top | Personalized, position-aware ranking models |


### 2.5 Expected Behavior Across Models

| Scenario | Precision | Recall | NDCG | Interpretation |
|:--|:--:|:--:|:--:|:--|
| Recommends a few very accurate items | High | Low | Moderate | High precision but misses many relevant items |
| Recommends many relevant items (some low-ranked) | Moderate | High | Low | Broad coverage, but poor ordering |
| Recommends relevant items near top consistently | High | High | High | Ideal recommender — strong ranking and coverage |
| Recommends irrelevant popular items | Low | Low | Low | Poor recommender — lacks personalization |


## 3. Per-User Metric Functions

### 3.1 Objective

Now that we understand the intuition behind the metrics, it’s time to **implement the core** functions that calculate them at a per-user level.
Each function will take a user’s predicted ranked list of items and their ground-truth positives, and return a numeric score for a given K.

### 3.2 Why This Step Matters

All downstream evaluation — whether for baselines, MF, or LTR models — depends on these building blocks.
By first defining clear, **unit-tested per-user functions**, we ensure:
1. Consistency across all model evaluations.
1. No hidden bugs in metric logic.
1. Reusability in future notebooks (batch evaluator, leaderboard, etc.).

### 3.3 Implementation Steps

#### 3.3.1 Define Precision@K

In [ ]:
def precision_at_k(pred_items, true_items, k):
    """
    Compute Precision@K for a single user.

    Parameters:
        pred_items (list): Ranked list of item IDs recommended to the user.
        true_items (set or list): Ground-truth relevant item IDs for the user.
        k (int): Cutoff rank position.

    Returns:
        float: Precision@K score.
    """
    if not pred_items or not true_items:
        return 0.0

    pred_k = pred_items[:k]                           # take top-K items
    hits = len(set(pred_k) & set(true_items))         # count correct ones
    precision = hits / k                              # fraction of top-K that are relevant
    return precision

#### 3.3.2 Define Recall@K

In [ ]:
def recall_at_k(pred_items, true_items, k):
    """
    Compute Recall@K for a single user.

    Parameters:
        pred_items (list): Ranked list of item IDs recommended to the user.
        true_items (set or list): Ground-truth relevant item IDs for the user.
        k (int): Cutoff rank position.

    Returns:
        float: Recall@K score.
    """
    if not pred_items or not true_items:
        return 0.0

    pred_k = pred_items[:k]
    hits = len(set(pred_k) & set(true_items))
    recall = hits / len(true_items)                   # fraction of relevant items retrieved
    return recall

#### 3.3.3 Define NDCG@K

In [ ]:
import numpy as np

def ndcg_at_k(pred_items, true_items, k):
    """
    Compute Normalized Discounted Cumulative Gain (NDCG@K) for a single user.

    Parameters:
        pred_items (list): Ranked list of item IDs recommended to the user.
        true_items (set or list): Ground-truth relevant item IDs for the user.
        k (int): Cutoff rank position.

    Returns:
        float: NDCG@K score.
    """
    if not pred_items or not true_items:
        return 0.0

    pred_k = pred_items[:k]
    rel = np.array([1 if item in true_items else 0 for item in pred_k])

    # Compute DCG
    dcg = np.sum(rel / np.log2(np.arange(2, len(rel) + 2)))

    # Compute IDCG (ideal DCG)
    ideal_rel = np.sort(rel)[::-1]
    idcg = np.sum(ideal_rel / np.log2(np.arange(2, len(ideal_rel) + 2)))

    if idcg == 0:
        return 0.0

    return dcg / idcg

### 3.4 Quick Sanity Check

Let’s verify that these functions behave as expected for a toy user.

In [ ]:
true_items = {1, 2, 3}
pred_items = [1, 4, 3, 7, 8]

for k in [3, 5]:
    print(f"K={k}: Precision={precision_at_k(pred_items, true_items, k):.2f}, "
          f"Recall={recall_at_k(pred_items, true_items, k):.2f}, "
          f"NDCG={ndcg_at_k(pred_items, true_items, k):.2f}")

## 4. Batch Evaluation Utility

### 4.1 Objective

Wrap the per-user metrics into a **batch evaluator** that:
1. accepts user_to_preds, user_to_truth, and a list of K values,
1. computes **macro-averaged** Precision@K, Recall@K, NDCG@K across users,
1. returns a clean, reusable result structure.

### 4.2 Why This Step Matters

Models produce predictions for many users. We need a **single call** that:
1. filters to users we can evaluate (have predictions and ground truth),
1. applies per-user metrics consistently,
1. aggregates results for reporting/leaderboards.

### 4.3 Implementation Steps

#### 4.3.1 Helper: normalize inputs

In [ ]:
def _to_set(x):
    # Accept list/set/array; return a set for O(1) membership checks
    return set(x) if not isinstance(x, set) else x

#### 4.3.2 Per-user wrapper (single K)

In [ ]:
def _user_metrics_at_k(pred_items, true_items, k):
    true_set = _to_set(true_items)
    p = precision_at_k(pred_items, true_set, k)
    r = recall_at_k(pred_items, true_set, k)
    n = ndcg_at_k(pred_items, true_set, k)
    return p, r, n

#### 4.3.3 Batch evaluator (macro average over users)

In [ ]:
import numpy as np

def evaluate_at_k(user_to_preds, user_to_truth, k_list, *,
                  include_precision_when_no_truth=False):
    """
    Macro-averaged metrics over users for each K in k_list.

    Policy:
      - Users with NO ground-truth positives are EXCLUDED from Recall/NDCG.
      - For Precision:
          include_precision_when_no_truth=False (default): exclude these users.
          If True, include them with precision=0 (stricter).
    Returns:
      dict: {"precision": {k: float}, "recall": {k: float}, "ndcg": {k: float}, "n_users_eval": {k: int}}
    """
    k_list = list(sorted(set(int(k) for k in k_list)))
    out = {"precision": {}, "recall": {}, "ndcg": {}, "n_users_eval": {}}

    # Users eligible for evaluation (must have predictions; truth may be empty)
    users_common = [u for u in user_to_preds.keys() if u in user_to_truth]

    for k in k_list:
        p_vals, r_vals, n_vals = [], [], []

        for u in users_common:
            preds = user_to_preds.get(u, [])
            truth = user_to_truth.get(u, [])

            has_truth = bool(truth) and len(truth) > 0
            if not has_truth:
                # Precision: include or skip based on policy
                if include_precision_when_no_truth:
                    p_vals.append(precision_at_k(preds, set(), k))  # will be 0.0
                # Recall/NDCG undefined → skip
                continue

            p, r, n = _user_metrics_at_k(preds, truth, k)
            p_vals.append(p)
            r_vals.append(r)
            n_vals.append(n)

        # Safe means (empty → 0.0)
        def _avg(xs):
            return float(np.mean(xs)) if len(xs) else 0.0

        out["precision"][k] = _avg(p_vals)
        out["recall"][k]    = _avg(r_vals)
        out["ndcg"][k]      = _avg(n_vals)
        out["n_users_eval"][k] = len(r_vals)  # users with non-empty truth

    return out

#### 4.3.4 Pretty print (optional)

In [ ]:
def print_metrics_table(results):
    ks = sorted(results["precision"].keys())
    header = f"{'K':>3} | {'Precision':>9} | {'Recall':>7} | {'NDCG':>6} | {'Users':>5}"
    print(header)
    print("-" * len(header))
    for k in ks:
        print(f"{k:>3} | {results['precision'][k]:>9.3f} | {results['recall'][k]:>7.3f} | "
              f"{results['ndcg'][k]:>6.3f} | {results['n_users_eval'][k]:>5d}")

### 4.4 Quick Sanity (toy example)

In [ ]:
# Tiny toy data (replace later with real preds from your model)
user_to_truth = {
    1: {10, 20, 30},
    2: {5},
    3: set(),          # no ground truth → excluded from recall/ndcg
}
user_to_preds = {
    1: [20, 99, 10, 8, 30],
    2: [7, 5, 6],
    3: [1, 2, 3],
}

res = evaluate_at_k(user_to_preds, user_to_truth, k_list=[3, 5])
print_metrics_table(res)

## 5. Sanity Tests (Toy, Self-Checking)

### 5.1 Objective

Verify each metric behaves as expected on tiny, known cases before using real MovieLens splits.

### 5.2 Tests

#### 5.2.1 Perfect ranking (all relevant at top)

In [ ]:
true = {1, 2, 3}
pred = [1, 2, 3, 9, 8]

assert abs(precision_at_k(pred, true, 3) - 1.0) < 1e-9
assert abs(recall_at_k(pred, true, 3)    - 1.0) < 1e-9
assert abs(ndcg_at_k(pred, true, 3)      - 1.0) < 1e-9
print("✅ Perfect ranking passes.")

#### 5.2.2 None relevant (zero scores)

In [ ]:
true = {4, 5}
pred = [1, 2, 3]
assert precision_at_k(pred, true, 3) == 0.0
assert recall_at_k(pred, true, 3)    == 0.0
assert ndcg_at_k(pred, true, 3)      == 0.0
print("✅ No-relevance case passes.")

#### 5.2.3 Mixed case (check ordering impact on NDCG)

In [ ]:
true = {10, 20}
pred_good = [10, 7, 20, 6, 5]  # both present, one at rank 1 → higher NDCG
pred_bad  = [7, 10, 5, 20, 6]  # both present, but lower positions → lower NDCG

p1, r1, n1 = precision_at_k(pred_good, true, 5), recall_at_k(pred_good, true, 5), ndcg_at_k(pred_good, true, 5)
p2, r2, n2 = precision_at_k(pred_bad,  true, 5), recall_at_k(pred_bad,  true, 5), ndcg_at_k(pred_bad,  true, 5)

assert p1 == p2 == 2/5 and r1 == r2 == 1.0 and n1 > n2
print("✅ Ordering affects NDCG (as expected).")

#### 5.2.4 Batch evaluator shape/policies

In [ ]:
user_to_truth = {1: {10,20},  2: {5},   3: set()}  # user 3 excluded from recall/ndcg
user_to_preds = {1: [20,10],  2: [1,5], 3: []}

res = evaluate_at_k(user_to_preds, user_to_truth, k_list=[1,2])
assert set(res.keys()) == {"precision", "recall", "ndcg", "n_users_eval"}
assert res["n_users_eval"][1] == 2 and res["n_users_eval"][2] == 2
print("✅ Batch evaluator policies OK.")

## 6. Run on MovieLens Validation/Test

### 6.1 Objective

Evaluate your recommendation outputs on **val/test** splits built earlier.

### 6.2 Build Ground Truth (if not already)

In [ ]:
# From earlier: val_df, test_df loaded in Configuration
user_to_truth_val  = val_df.groupby("UserID")["MovieID"].apply(set).to_dict()
user_to_truth_test = test_df.groupby("UserID")["MovieID"].apply(set).to_dict()

len(user_to_truth_val), len(user_to_truth_test)

### 6.3 Plug in Predictions

(Replace placeholders with your model’s ranked outputs.)

In [ ]:
# Example placeholders — REPLACE with real predictions
# Format: {user_id: [ranked_item_ids ...]}
user_to_preds_val  = {}  # e.g., built by your baseline/MF/LTR notebook
user_to_preds_test = {}

user_to_preds_val = val_df.groupby("UserID")["MovieID"].apply(set).to_dict()
user_to_preds_test = test_df.groupby("UserID")["MovieID"].apply(set).to_dict()
assert len(user_to_preds_val)  > 0, "Provide validation predictions."
assert len(user_to_preds_test) > 0, "Provide test predictions."

### 6.4 Evaluate and Print Tables

In [ ]:
K_LIST = [5, 10, 20]


# This code is added later to convert set to a list

user_to_preds_val = {
    u: list(items)
    for u, items in user_to_preds_val.items()
}

user_to_preds_test = {
    u: list(items)
    for u, items in user_to_preds_test.items()
}
#
val_results  = evaluate_at_k(user_to_preds_val,  user_to_truth_val,  K_LIST)
test_results = evaluate_at_k(user_to_preds_test, user_to_truth_test, K_LIST)

print("=== Validation ===")
print_metrics_table(val_results)
print("\n=== Test ===")
print_metrics_table(test_results)

### 6.5 (Optional) Save Results

In [ ]:
import json, os
os.makedirs("metrics_out", exist_ok=True)

with open("metrics_out/val_results.json", "w") as f:
    json.dump(val_results, f, indent=2)
with open("metrics_out/test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("✅ Saved metrics to metrics_out/")

## 7. Result Summary & Next Steps

### 7.1 Objective

Wrap the evaluation in a compact summary table and (optionally) persist a simple **leaderboard** so future models can be compared consistently.

### 7.2 Compact Summary Table

In [ ]:
import pandas as pd

def results_to_df(name, res):
    ks = sorted(res["precision"].keys())
    rows = []
    for k in ks:
        rows.append({
            "split": name,
            "K": k,
            "Precision": round(res["precision"][k], 5),
            "Recall":    round(res["recall"][k], 5),
            "NDCG":      round(res["ndcg"][k], 5),
            "UsersEval": int(res["n_users_eval"][k]),
        })
    return pd.DataFrame(rows)

summary_val  = results_to_df("val",  val_results)
summary_test = results_to_df("test", test_results)

summary = pd.concat([summary_val, summary_test], ignore_index=True)
summary

### 7.3 Save/Append to Leaderboard

In [ ]:
import os

LB_PATH = "metrics_out/leaderboard.csv"
os.makedirs(os.path.dirname(LB_PATH), exist_ok=True)

# Add run metadata here if you want (model name, notes, seed, timestamp)
MODEL_NAME = "baseline_placeholder"  # e.g., "popularity@session", "als@factors=64"
RUN_NOTES  = "first metrics harness run"

summary_lb = summary.copy()
summary_lb["model"] = MODEL_NAME
summary_lb["notes"] = RUN_NOTES
summary_lb = summary_lb[["model","notes","split","K","Precision","Recall","NDCG","UsersEval"]]

# Append or create
if os.path.exists(LB_PATH):
    existing = pd.read_csv(LB_PATH)
    leaderboard = pd.concat([existing, summary_lb], ignore_index=True)
else:
    leaderboard = summary_lb

leaderboard.to_csv(LB_PATH, index=False)
print(f"✅ Leaderboard updated → {LB_PATH}")
leaderboard.tail(10)

### 7.4 Quick Visual Check (optional)

In [ ]:
# Simple pivot to compare metrics by K and split for this run
pivot = summary.pivot(index="K", columns="split", values=["Precision","Recall","NDCG"])
pivot